# Data Download

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile
import csv

In [ ]:
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = (
    CURRENT_DIR.parent 
    if CURRENT_DIR.name == "code"
    else CURRENT_DIR
)

DATA_DIR = PROJECT_ROOT / "data"
TLC_DIR = DATA_DIR / "tlc_data"
ZONE_DIR = DATA_DIR / "taxi_zones"
WEATHER_DIR = DATA_DIR / "weather"

DATA_DIR.mkdir(parents=True, exist_ok=True)
TLC_DIR.mkdir(parents=True, exist_ok=True)
WEATHER_DIR.mkdir(parents=True, exist_ok=True)
ZONE_DIR.mkdir(parents=True, exist_ok=True)

YEAR = 2024
MONTHS = range(1, 7)  # January to June

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

In [ ]:
def download_file(url, output_path):
    output_path = Path(output_path)

    if output_path.exists() and output_path.stat().st_size > 0:
        print(f"Already exists: {output_path.name}")
        return
    temporary_path = output_path.with_suffix(output_path.suffix + ".part")
    print(f"Downloading {output_path.name}...")

    try:
        urlretrieve(url, temporary_path)
        temporary_path.replace(output_path)
        print(f"Downloaded {output_path.name}")

    except Exception:
        if temporary_path.exists():
            temporary_path.unlink()
        raise

## Yellow Taxi

In [ ]:
TLC_BASE_URL = ("https://d37ci6vzurychx.cloudfront.net/trip-data/")

for month in MONTHS:
    filename = f"yellow_tripdata_{YEAR}-{month:02d}.parquet"
    url = f"{TLC_BASE_URL}{filename}"
    output_path = TLC_DIR / filename

    download_file(url, output_path)

## Taxi Zones

In [ ]:
ZONE_LOOKUP_URL = (
    "https://d37ci6vzurychx.cloudfront.net/misc/"
    "taxi_zone_lookup.csv"
)

ZONE_SHAPEFILE_URL = (
    "https://d37ci6vzurychx.cloudfront.net/"
    "misc/taxi_zones.zip"
)

ZONE_LOOKUP_PATH = ZONE_DIR / "taxi_zone_lookup.csv"
ZONE_ZIP_PATH = ZONE_DIR / "taxi_zones.zip"

download_file(ZONE_LOOKUP_URL, ZONE_LOOKUP_PATH)
download_file(ZONE_SHAPEFILE_URL, ZONE_ZIP_PATH)

if not list(ZONE_DIR.glob("*.shp")):
    with ZipFile(ZONE_ZIP_PATH, "r") as zone_zip:
        zone_zip.extractall(ZONE_DIR)
    print(f"Extracted: taxi zone shapefile")
else:
    print(f"Already extracted: taxi zone shapefile")

## Weather

In [ ]:
WEATHER_URLS = {
    "JFK": "https://www.ncei.noaa.gov/data/local-climatological-data/access/2024/74486094789.csv",
    "LGA": "https://www.ncei.noaa.gov/data/local-climatological-data/access/2024/72503014732.csv"
}

for airport, url in WEATHER_URLS.items():
    output_path = WEATHER_DIR / f"{airport.lower()}_weather_{YEAR}.csv"

    output_path = (
        WEATHER_DIR / f"{airport.lower()}_weather_{YEAR}.csv"
    )

    print(f"{airport}: {url}")
    download_file(url, output_path)